# Proximity-Weighted Fault Localization

This example provides an overview of proximity-weighted fault Localization (PWFL) method.
The PWFL approach is a fault localization technique that uses the proximity information of test cases to improve the accuracy of fault localization.
The method is based on the intuition that the lines in the test cases that execute closer to the fault location are more likely to be related to the fault.
PWFL assigns weights to test lines based on this idea and uses these weights to improve the accuracy of fault localization.

## The Example

In this example, we will demonstrate how to use the PWFL method to localize faults in a program.
The program we will investigate is the following one:

In [1]:
def middle(x, y, z):
    if y < z:
        if x < y:
            return y
        elif x < z:
            return y  # bug
    elif x > y:
        return y
    elif x > z:
        return x
    return z

This program contains a bug in line 6. The bug is that the program should return x instead of y.

## Statistical Fault Localization

Statistical fault localization is a technique that uses the information from the execution of test cases to identify the likely locations of faults in a program.
The technique correlates the execution of lines with the presence of faults and assigns suspiciousness scores to lines based on this correlation.
The suspiciousness scores are used to rank the lines of the program and identify the most likely locations of faults.
 

## Test Cases

To localize the fault in the program, we will use the following test cases:

In [2]:
def test_1():
    assert middle(1, 2, 3) == 2
    assert middle(3, 2, 1) == 2
    
def test_2():
    assert middle(3, 1, 2) == 2
    assert middle(2, 2, 1) == 2
    
def test_3():
    assert middle(2, 3, 1) == 2
    assert middle(2, 1, 3) == 2
    
def test_4():
    assert middle(2, 2, 3) == 2
    assert middle(1, 3, 2) == 2

## Instrumenting the Program and the Test Cases

To apply PWFL, we need to instrument the program and the test cases with SFLKit to collect the proximity information of the test cases.

### Writing the Program and the Test Cases to Files

First, we need to save the program to a file.

In [3]:
import sflkit
import inspect

We need the source code of the `middle` function.

In [4]:
source = inspect.getsource(middle)
print(source)

def middle(x, y, z):
    if y < z:
        if x < y:
            return y
        elif x < z:
            return y  # bug
    elif x > y:
        return y
    elif x > z:
        return x
    return z



Then we create a directory to save the program file and the test file.

In [5]:
from pathlib import Path
subject = Path("middle")
subject.mkdir(exist_ok=True)

Next, we save the program to a file.

In [6]:
middle_py = subject / 'middle.py'
with open(middle_py, 'w') as f:
    f.write(source)

We also need to save the test cases to a file.
First, we get the source code of the test cases and add an additional import statement to import the `middle` function.

In [7]:
test_source = (
        "from middle import middle\n\n" + 
        inspect.getsource(test_1) + "\n" + 
        inspect.getsource(test_2) + "\n" + 
        inspect.getsource(test_3) + "\n" + 
        inspect.getsource(test_4) + "\n"
)
print(test_source)

from middle import middle

def test_1():
    assert middle(1, 2, 3) == 2
    assert middle(3, 2, 1) == 2

def test_2():
    assert middle(3, 1, 2) == 2
    assert middle(2, 2, 1) == 2

def test_3():
    assert middle(2, 3, 1) == 2
    assert middle(2, 1, 3) == 2

def test_4():
    assert middle(2, 2, 3) == 2
    assert middle(1, 3, 2) == 2




Then we save the test cases to a file.

In [8]:
test_middle_py = subject / 'tests.py'
with open(test_middle_py, 'w') as f:
    f.write(test_source)

### Instrumentation

Let's instrument the program and the test cases with SFLKit.
We will use the `Tarantula` metric to calculate the suspiciousness scores of the lines.
For PWFL, we will use only the tests lines because it's the easiest to follow.
Let's create a configuration for the instrumentation.

In [9]:
config = sflkit.Config.create(
    path=str(subject), 
    working="middle_tmp", 
    language="python",
    predicates="line",
    metrics="Tarantula",
    test_events="test_line",
    exclude="tests.py",
    test_files="tests.py",
    runner="pytest_runner",
    mapping_path="middle_mapping.json",
)

We can now instrument the program and the test cases.

In [10]:
sflkit.instrument_config(config)

sflkit :: INFO     :: I found 10 events in middle/middle.py.
sflkit :: INFO     :: I found 9 events in middle/tests.py.
sflkit :: INFO     :: I found 19 events in middle.


## Running the Test Cases

Now that we have instrumented the program and the test cases, we can run the test cases to collect the events from SFLKit.

In [11]:
middle_events = Path("middle_events")
runner = config.runner.runner()
runner.run(config.instrument_working, middle_events, files=["tests.py"])

The events of the test cases are saved in the `middle_events` directory.
Let's check if the events are saved correctly.

In [12]:
test_1_events = middle_events / "passing" / "tests_py__test_1"
test_2_events = middle_events / "passing" / "tests_py__test_2"
test_3_events = middle_events / "failing" / "tests_py__test_3"
test_4_events = middle_events / "passing" / "tests_py__test_4"

assert all(e.exists() for e in [test_1_events, test_2_events, test_3_events, test_4_events])

For the analysis, we need to create event files from the events.

In [13]:
from sflkit.events.event_file import EventFile
from sflkit.events.mapping import EventMapping

event_mapping = EventMapping.load(config)
test_1_events = EventFile(test_1_events, 1, event_mapping)
test_2_events = EventFile(test_2_events, 2, event_mapping)
test_3_events = EventFile(test_3_events, 3, event_mapping, failing=True)
test_4_events = EventFile(test_4_events, 4, event_mapping)

## Getting the Results of Statistical Fault Localization

Now that we have the events of the test cases, we can use the SFL methods to localize the fault in the program.

In [14]:
from sflkit.analysis.factory import LineFactory

analyzer = sflkit.Analyzer([test_3_events], [test_1_events, test_2_events, test_4_events], factory=LineFactory())
analyzer.analyze()

The following code allows us to visualize the results of the statistical fault localization.

In [15]:
from IPython.core.display import HTML
from sflkit.color import ColorCode

def colorize(suggestions):
    code = ColorCode(suggestions)
    return code.code("middle.py", source, color=True, suspiciousness=True)

Let's visualize the results of the statistical fault localization.

In [16]:
from sflkit.analysis.spectra import Spectrum
from sflkit.analysis.analysis_type import AnalysisType

HTML(colorize(analyzer.get_sorted_suggestions(subject, Spectrum.Tarantula, AnalysisType.LINE)))

We can see that the statistical fault localization method suggests that line 6 and 10 are the most suspicious lines.
However, since the actual fault is in line 6, the method does not provide a definitive answer about the location of the fault.

## Proximity-Weighted Fault Localization

To improve the accuracy of fault localization, we will use the PWFL method.
We again need to analyze the events of the test cases to calculate the weights of the test cases.
Additionally, we need to provide PWFL with a model to calculate the weights.
In this example, we will use the `TestLineModel` model, which assigns weights to test cases based on the proximity information derived from lines of the test cases.

In [17]:
from sflkit.weights import ProximityAnalyzer, TestLineModel

analyzer = ProximityAnalyzer(
    TestLineModel, 
    [test_3_events], 
    [test_1_events, test_2_events, test_4_events], 
    factory=LineFactory()
)
analyzer.analyze()

Let's visualize the results of the proximity-weighted fault localization.

In [18]:
HTML(colorize(analyzer.get_sorted_suggestions(subject, Spectrum.Tarantula, AnalysisType.LINE, use_weight=True)))

As we can see, the PWFL method suggests that line 6 is the most suspicious line, since it is more important for the failing test case based on the failing assertion.